# Generalized Additive Models (GAMs) in Python: A Practical Tutorial

Welcome to the hands-on tutorial for **magli**!

In this notebook, you will learn:
1. **Why GAMs?** The failure of linear lines and polynomial wiggles.
2. **How Penalized Splines Work:** Elastic springs and the sweet spot.
3. **Understanding Model Output:** Reading EDF, F-tests, and p-values.
4. **Confidence vs. Prediction Intervals:** What uncertainty really means.
5. **Multi-Term Formulas & Additive Decomposition:** Isolating individual effects.
6. **2-D Interactions via Tensor Products:** Beyond simple additivity.
7. **Shape Constraints:** Monotonicity and periodicity from physics.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from magli import GAM, LinGAM, GAMCore

# Set random seed for reproducibility
np.random.seed(42)
print('magli imported successfully!')


## 1. The Dilemma: Underfitting vs. Overfitting

Let us generate noisy observations from a nonlinear ground-truth function:
$$y = \sin(2\pi x) + \varepsilon, \quad \varepsilon \sim \mathcal{N}(0, 0.25^2)$$


In [ ]:
n = 120
x = np.sort(np.random.uniform(0, 1, n))
y_true = np.sin(2 * np.pi * x)
y = y_true + np.random.normal(0, 0.25, n)

# Compare: Linear line vs. High-degree polynomial vs. Penalized GAM
x_eval = np.linspace(0, 1, 300)

# 1. Linear model
poly_1 = np.poly1d(np.polyfit(x, y, deg=1))

# 2. Degree-12 polynomial (notice Runge's boundary oscillations!)
poly_12 = np.poly1d(np.polyfit(x, y, deg=12))

# 3. magli GAM with automatic REML smoothing
gam = GAM(k=12, bs='cr', backend='wood').fit(x, y)
y_gam = gam.predict(x_eval)

plt.figure(figsize=(10, 5))
plt.scatter(x, y, c='gray', alpha=0.6, s=25, label='Noisy Data')
plt.plot(x_eval, np.sin(2 * np.pi * x_eval), 'k--', lw=2, label='True Signal')
plt.plot(x_eval, poly_1(x_eval), 'tab:red', lw=1.8, label='Linear (Underfit)')
plt.plot(x_eval, poly_12(x_eval), 'tab:orange', lw=1.8, label='Degree-12 Polynomial (Wild Overfit)')
plt.plot(x_eval, y_gam, 'tab:blue', lw=2.5, label='magli GAM (Optimal REML)')
plt.ylim(-2.0, 2.0)
plt.title('Why GAMs? Balancing Flexibility and Regularity', fontweight='bold')
plt.xlabel('x')
plt.ylabel('y')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


## 2. Understanding Effective Degrees of Freedom (EDF)

How does `magli` avoid overfitting with 12 basis functions?
The penalty $\lambda S$ acts like elastic springs between neighboring spline coefficients.

Let's inspect the estimated model summary:


In [ ]:
print(gam.summary())
print(f'Estimated optimal smoothing parameter (lambda): {gam.lam:.4f}')
print(f'Effective Degrees of Freedom (EDF):             {gam.statistics_["edof"]:.2f} (out of k={gam.n_splines})')
print(f'Deviance Explained:                             {gam.deviance_explained() * 100:.1f}%')


### The Rule of Thumb on EDF:
- **EDF = 1.0**: Collapsed to a pure linear slope.
- **EDF = 2.0**: A pure parabola.
- **EDF ≈ 5.2**: The true sine wave shape requires ~5 effective parameters.
- If **EDF $\approx k - 1$**, your basis is saturated and you should increase `k`.


## 3. Confidence Intervals vs. Prediction Intervals

- **Confidence Interval (width=0.95)**: Uncertainty of the **mean curve** $\hat{f}(x)$.
- **Prediction Interval (width=0.95)**: Where a **future individual observation** $y_{\text{new}}$ will fall (includes measurement noise $\sigma^2$).


In [ ]:
ci = gam.confidence_intervals(x_eval, width=0.95)
pi = gam.prediction_intervals(x_eval, width=0.95)

plt.figure(figsize=(9, 4.8))
plt.scatter(x, y, c='gray', alpha=0.5, s=20, label='Data Points')
plt.plot(x_eval, y_gam, 'tab:blue', lw=2, label='Fitted GAM Mean Curve')
plt.fill_between(x_eval, ci[:, 0], ci[:, 1], color='tab:blue', alpha=0.3, label='95% Confidence Interval (Mean)')
plt.fill_between(x_eval, pi[:, 0], pi[:, 1], color='tab:blue', alpha=0.12, label='95% Prediction Interval (New Data)')
plt.title('Uncertainty Quantification in magli', fontweight='bold')
plt.xlabel('x')
plt.ylabel('y')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


## 4. Multi-Term Formulas & Additive Decomposition

In real engineering and science, data has multiple inputs. GAMs decompose complex multivariate functions into human-interpretable 1D and 2D curves:
$$\hat{y} = \beta_0 + f_1(x_0) + f_2(x_1) + \beta_3 x_2$$


In [ ]:
# Simulate multi-feature dataset
N = 250
X = np.random.uniform(0, 10, (N, 3))
y_multi = np.sin(X[:, 0]) + 0.4 * (X[:, 1] - 5)**2 + 1.5 * X[:, 2] + np.random.normal(0, 0.3, N)

# Fit multi-term model: 2 smooth terms + 1 linear term
multi_gam = GAMCore("s(0, k=10) + s(1, k=10) + l(2)", backend="wood")
multi_gam.fit(X, y_multi)

print(multi_gam.summary())

# Plot additive decomposition (each component in isolation!)
multi_gam.plot_decomposition(X, y_multi)
plt.show()


## 5. Capturing Interactions via 2-D Tensor Products (`te`)

When two variables synergize (e.g. Temperature $\times$ Vibration), additivity $f(x_1) + f(x_2)$ is insufficient.
Use `te(0, 1)` to build a flexible 2D sheet with independent coordinate smoothing:


In [ ]:
# Simulate interactive surface (saddle function)
X_surf = np.random.uniform(0, 1, (200, 2))
y_surf = np.sin(np.pi * X_surf[:, 0]) * np.cos(np.pi * X_surf[:, 1]) + np.random.normal(0, 0.1, 200)

te_model = GAMCore("te(0, 1, k=5, bs='cr')", backend="wood").fit(X_surf, y_surf)

# Evaluate on a 2D grid
grid_axis = np.linspace(0, 1, 40)
GX1, GX2 = np.meshgrid(grid_axis, grid_axis)
G_pts = np.column_stack([GX1.ravel(), GX2.ravel()])
Z_pred = te_model.predict(G_pts).reshape(GX1.shape)

fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection='3d')
ax.plot_surface(GX1, GX2, Z_pred, cmap='viridis', alpha=0.8, edgecolor='none')
ax.scatter(X_surf[:, 0], X_surf[:, 1], y_surf, color='crimson', s=12, label='Data points')
ax.set_title('Recovered 2D Interaction Surface via te(0, 1)', fontweight='bold')
ax.set_xlabel('X[0]')
ax.set_ylabel('X[1]')
ax.set_zlabel('y')
plt.legend()
plt.show()


## 6. Enforcing Domain Constraints (Monotonicity & Periodicity)

In physical applications (dose-response curves, wear degradation, diurnal cycles), physics dictates shape.
The **Eilers-Marx backend** allows you to enforce soft shape constraints:


In [ ]:
# Dose-response curve: response CANNOT decrease with higher dose
dose = np.linspace(0, 10, 80)
resp_true = 5.0 * (1.0 - np.exp(-0.4 * dose))
resp_noisy = resp_true + np.random.normal(0, 0.4, len(dose))

# 1. Unconstrained GAM (might dip due to noise)
uncon_gam = GAM(n_splines=12, backend="eilers").fit(dose, resp_noisy)

# 2. Monotonically increasing GAM (guarantees non-negative derivative everywhere)
mono_gam = GAM(n_splines=12, constraint="mono_inc", backend="eilers").fit(dose, resp_noisy)

plt.figure(figsize=(9, 4.8))
plt.scatter(dose, resp_noisy, c='gray', alpha=0.6, label='Noisy Measurements')
plt.plot(dose, uncon_gam.predict(dose), 'tab:red', ls='--', lw=2, label='Unconstrained (Local noise dips)')
plt.plot(dose, mono_gam.predict(dose), 'tab:green', lw=2.5, label='Monotonic Increasing (Physically valid)')
plt.title('Enforcing Physical Domain Constraints', fontweight='bold')
plt.xlabel('Dose')
plt.ylabel('Response')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


## Summary & Next Steps

Congratulations! You now understand the core machinery of Generalized Additive Models:
1. **Wood backend (`backend="wood"`)**: Default choice for general regression, automatic continuous REML tuning, and rigorous hypothesis testing.
2. **Eilers-Marx backend (`backend="eilers"`)**: Choice when physical domain constraints (monotonicity, convexity) are required.
3. **Additive decomposition**: Inspecting isolated partial effects via `plot_decomposition()`.

For full documentation, visit: **https://dacts23.github.io/magli/**
